# Dispersion Pricing — API Demo

Solve and price corridor variance swaps through the public `solve()` / `price()` API,
and access the generated FPF trade representations programmatically.

**Covered here:** mono-corridor solve · cross-corridor solve (LV, and LV+LSV+LCM) ·
individual correlations · capped vs uncapped · price at given strikes · FPF strings &
programmatic FPF building · discount factor & vanilla-strike columns.

> Requires a Bloomberg session and the pricing services. Edit the **Inputs** cell and
> re-run top-to-bottom. Input tickers accept **RIC or BBG** forms
> (`ISP.MI` / `.STOXX50E` or `ISP IM Equity` / `SX5E Index`).


## Setup

Locates the `functions` package (set `GAIA_REPO` to your repo root if not found).

In [ ]:
import os, sys
from pathlib import Path

def _find_repo_root() -> Path:
    candidates = [os.environ.get("GAIA_REPO"), os.path.abspath("../.."), os.getcwd(),
                  os.path.abspath(".."), os.path.expanduser("~/Disp")]
    for c in candidates:
        if c and (Path(c) / "functions" / "dispersion").is_dir():
            return Path(c)
    raise FileNotFoundError(
        "Could not locate the 'functions/dispersion' package. "
        "Set GAIA_REPO to your repo root.")

sys.path.insert(0, str(_find_repo_root()))

import pandas as pd
from datetime import date, timedelta

from functions.dispersion import solve, price, DispersionConfig
from functions.dispersion.models import ProductType

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

print("Setup OK — repo:", _find_repo_root())

## 1. Inputs

Two shapes of input table, both with **RIC-style** tickers:

| Column | Mono corridor | Cross corridor |
|--------|:---:|:---:|
| `Variance Asset` | the stock | the **index** |
| `Corridor Condition Asset` | the stock (same) | the stock |
| `Currency` | ✓ | ✓ |
| `Strike Cross Corridor (%)` | — | `price()` only |
| `Strike Mono Var Swap (%)` | `price()` only | `price()` only |
| `Correlation` | — | Individual mode only (percent) |

Key `solve()` / `price()` parameters: `eqeq_lambda` (EQ-EQ correl decay),
`eqfx_shift` (FX correl shift), `vol_mode` ('ATMF'/'ATMS'),
`correl_input_method` ('Global Parameters'/'Individual Correlations'),
`use_lsv`+`lsv_params`, `use_lcm`+`lcm_properties` (cross only).

In [ ]:
# ── Dates (edit me) ──
strike_date   = date.today()
last_obs_date = strike_date + timedelta(days=365)

# ── Universe (edit me) ──
cross_df = pd.DataFrame({
    "Variance Asset": [".STOXX50E", ".STOXX50E", ".STOXX50E"],
    "Corridor Condition Asset": ["TTEF.PA", "ASML.AS", "SASY.PA"],
    "Currency": ["EUR", "EUR", "EUR"],
})
mono_df = pd.DataFrame({
    "Variance Asset": ["TTEF.PA", "ASML.AS", "SASY.PA"],
    "Corridor Condition Asset": ["TTEF.PA", "ASML.AS", "SASY.PA"],
    "Currency": ["EUR", "EUR", "EUR"],
})

# ── Config (edit me) ──
config_cross = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=True,
    barrier_up=1.3, barrier_down=0.7,
    local_cap=2.5, is_capped=True,
)
config_mono = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=False,
    barrier_up=1.3, barrier_down=0.7,
    local_cap=2.5, is_capped=True,
)

# ── LSV params (indexed by RIC; missing names get stock defaults) ──
lsv_df = pd.DataFrame({
    "RIC": ["TTEF.PA", "ASML.AS", "SASY.PA"],
    "VolOfVar": [0.7, 0.7, 0.7],
    "Eq/VolCorrel": [-0.7, -0.7, -0.7],
    "MeanReversion": [2.0, 2.0, 2.0],
}).set_index("RIC")

# ── LCM properties (cross-corridor only) ──
lcm_props = {
    "AggregatorType": "Basket",
    "CallSkew": [-0.5],
    "PutSkew": [-0.9],
    "LambdaAtm": [0.2368839],
    "LambdaFromRho0": 0.1,
    "LambdaPricing": 0.53,
}

print(f"strike_date={strike_date}  last_obs_date={last_obs_date}")

## 2. Solve — Mono Corridor

`Variance Asset == Corridor Condition Asset`. The LV strike comes out as
`Strike Mono Corr (%)`; with `is_capped=True` also the real-priced capped variant.

In [ ]:
result_mono = solve(
    df=mono_df, config=config_mono,
    last_obs_date=last_obs_date, strike_date=strike_date,
    eqeq_lambda=0.10, eqfx_shift=-0.05, vol_mode="ATMF",
)
print("Success:", result_mono.success)
result_mono.results_df

## 3. Solve — Cross Corridor

`Variance Asset` (index) ≠ `Corridor Condition Asset` (stock). Two cases:
**LV only** (base model) and **LV + LSV + LCM** (all adjustments on).

### 3A. LV only

In [ ]:
result_lv = solve(
    df=cross_df, config=config_cross,
    last_obs_date=last_obs_date, strike_date=strike_date,
    eqeq_lambda=0.10, eqfx_shift=-0.05, vol_mode="ATMF",
    correl_input_method="Global Parameters",
)
print("Success:", result_lv.success)
result_lv.results_df

### 3B. LV + LSV + LCM

Both adjustments at once: `Strike Cross Corr LSV (%)` and `Strike Cross Corr LCM (%)`
appear next to the LV strike. With `is_capped=True`, all cap-priced variants
(`... Cap Priced LV/LSV/LCM`) are populated via the internal Phase-2b re-pricing.
(LCM is **cross-corridor only**. LSV params resolve: RIC in `lsv_params` → your row;
known index → built-in; otherwise → stock defaults.)

In [ ]:
result_full = solve(
    df=cross_df, config=config_cross,
    last_obs_date=last_obs_date, strike_date=strike_date,
    eqeq_lambda=0.10, eqfx_shift=-0.05, vol_mode="ATMF",
    correl_input_method="Global Parameters",
    use_lsv=True, lsv_params=lsv_df,
    use_lcm=True, lcm_properties=lcm_props,
)
print("Success:", result_full.success)
result_full.results_df

### 3C. Side-by-side comparison

In [ ]:
def _strikes(res, label):
    df = res.results_df
    cols = [c for c in df.columns if "Strike" in c]
    out = df[cols].copy()
    out.index = [f"{label} #{i}" for i in range(len(out))]
    return out

pd.concat([_strikes(result_lv, "LV"), _strikes(result_full, "LV+LSV+LCM")])

## 4. Individual Correlations

Override the model EQ-EQ correlation per ticker with a `Correlation` column
(**percent**, e.g. 55 = 55%) + `correl_input_method="Individual Correlations"`.

In [ ]:
cross_df_indiv = cross_df.copy()
cross_df_indiv["Correlation"] = [55, 60, 48]  # percent, one per ticker

result_indiv = solve(
    df=cross_df_indiv, config=config_cross,
    last_obs_date=last_obs_date, strike_date=strike_date,
    eqeq_lambda=0.10, eqfx_shift=-0.05, vol_mode="ATMF",
    correl_input_method="Individual Correlations",
)
print("Success:", result_indiv.success)
result_indiv.results_df

## 5. Capped vs Uncapped

`is_capped=True` runs a **two-phase** cap computation: theoretical cap impact, then a
real re-price at the cap-adjusted strike (columns `Cap Theoretical Impact`,
`Strike Cross Corr Cap Adjusted`, `... Cap Priced LV/LSV/LCM`). With `is_capped=False`
only the uncapped strikes are produced — the same flag the backtester/optimizer now
honour for the per-leg local cap.

In [ ]:
config_uncapped = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=True,
    barrier_up=1.3, barrier_down=0.7,
    local_cap=2.5, is_capped=False,      # no cap columns, uncapped strikes only
)

result_uncapped = solve(
    df=cross_df, config=config_uncapped,
    last_obs_date=last_obs_date, strike_date=strike_date,
    eqeq_lambda=0.10, eqfx_shift=-0.05, vol_mode="ATMF",
)
print("Success:", result_uncapped.success)

cap_cols = [c for c in result_lv.results_df.columns if "Cap" in c]
print("Cap columns produced when is_capped=True:")
display(result_lv.results_df[cap_cols] if cap_cols else pd.DataFrame())

## 6. Price — at Given Strikes

`price()` evaluates the structure at **your** strikes (percent in the input table).
LSV/LCM optional and combinable, exactly like `solve()`.

In [ ]:
price_df = cross_df.copy()
price_df["Strike Cross Corridor (%)"] = [22.5, 28.1, 19.4]
price_df["Strike Mono Var Swap (%)"] = [19.8, 24.3, 17.2]

price_result = price(
    df=price_df, config=config_cross,
    last_obs_date=last_obs_date, strike_date=strike_date,
    eqeq_lambda=0.10, eqfx_shift=-0.05, vol_mode="ATMF",
    correl_input_method="Global Parameters",
    use_lsv=True, lsv_params=lsv_df,
    use_lcm=True, lcm_properties=lcm_props,
)
print("Success:", price_result.success)
price_result.results_df

## 7. FPF Access

Two ways to the FPF trade representations:

- **from results**: every `TickerSolveResult` carries its FPF strings (cross, mono,
  and capped variants when `is_capped=True`)
- **programmatic**: build one reference FPF via `build_corridor_fpf`, then **clone**
  it for other tickers (instant, no extra service calls) — the pattern the batch
  solver uses internally

In [ ]:
# ── 7A. FPF strings from a solve result ──
r = result_lv.ticker_results[0]
for attr in ["fpf_string_cross", "fpf_string_mono", "fpf_string_cap_lv",
             "fpf_string_cap_lsv", "fpf_string_cap_lcm"]:
    s = getattr(r, attr, None)
    if s:
        print(f"{attr}: {s[:110]}...")

In [ ]:
# ── 7B. Programmatic batch build: one reference FPF, then clones ──
from functions.dispersion._pricing import build_corridor_fpf
from fpf_builder_utils.corridorCovarianceSwap_v4 import (
    FPFUnifiedEconomicsWrapper, corridorCovarianceSwap_v4,
)

ref_ticker, corr_asset = "TTEF.PA", ".STOXX50E"

ref_fpf_str = build_corridor_fpf(
    tickers=[ref_ticker],
    last_obs_date=last_obs_date, strike_date=strike_date,
    strikes=[0.000001],          # placeholder — we only need the schedule
    weights=[1.0],
    low_barrier=0.7, high_barrier=1.3,
    is_capped=False,
    corr_asset=corr_asset,
    schedule_calendar_asset=ref_ticker,
    currency="EUR",
    use_parameters=False,
)
ref_obj = FPFUnifiedEconomicsWrapper.from_data(ref_fpf_str)

# Clone for the other tickers — instant, no HTTP
fpf_strings = {}
for t in ["ASML.AS", "SASY.PA"]:
    cloned = corridorCovarianceSwap_v4(
        ref_obj, ticker=t, strike=0.000001,
        low_barrier=0.7, high_barrier=1.3, corr_asset=corr_asset,
    )
    fpf_strings[t] = cloned.to_fpf_string() if hasattr(cloned, "to_fpf_string") else str(cloned)

print({k: v[:80] + "..." for k, v in fpf_strings.items()})

---
## Output Columns Reference (cross-corridor, `is_capped=True`)

| Column | Description |
|--------|-------------|
| `Strike Cross Corr (%)` | LV strike = sqrt(−EV/RA), RA = ZCB × E[n_corridor_obs]/n_total |
| `Strike Cross Corr LSV/LCM (%)` | LSV / LCM-adjusted strikes (uncapped) |
| `Strike Cross Corr Cap Priced LV/LSV/LCM (%)` | real-priced capped variants |
| `EV Cross`, `EV Cross LSV`, `EV Cross LCM` | expected-variance values (%) |
| `RA Cross` / `RA Mono` | range accrual = **undiscounted** day fraction E[n_corridor_obs]/n_total (the ZCB only enters the strike) |
| `Discount Factor` | unfunded ZCB of the leg currency (100% reoffer) |
| `Barrier Down/Up (%)` | corridor barriers |
| `Tenor (bd)` | observation days in the FPF schedule |
| `Strike Vanilla Var (%)` | as-if-vanilla strike sqrt(−EV/ZCB) of the corridor EV |
| `EV Mono`, `Strike Mono Corr (%)` | mono-corridor EV and strike |
| `Cap Theoretical Impact`, `Strike Cross Corr Cap Adjusted` | phase-1 cap diagnostics |
| `Obs Dates Cross/Mono` | business days in the observation schedules |
| `ATMF Vol` / `ATMS Vol` | implied vols used (matched per asset) |
| `Correlation` | EQ-EQ correlation from the model (or your column) |

**Mono-corridor FPF columns:** `FPF LV Uncapped`, `FPF LSV Uncapped`,
`FPF LCM Uncapped`, `FPF LV Cap`, `FPF LSV Cap`, `FPF LCM Cap` — shown only
when non-empty (mono mode has a single structure, no Cross/Mono split).

**Conventions:** strikes in **percent** in the input tables; **RIC or BBG**
tickers both accepted; LCM is cross-corridor only; `Correlation` in percent;
`is_capped` drives both the cap columns and the FPF variants; RA is computed
via the `PayoutTraceVariableExpectation` metric (no dedicated RA instrument).
